# Real Time Clock

*Written by Grace Lo & Alina Wang*

The Real Time Clock (RTC) is directly implemented in the `SD_Card_Methane` library to accurately keep track of time and make note of when data is logged. The RP2040 has built-in RTC functionality, so additional hardware is not necessary. 

## API

For the `SD_Card_Methane` library, the initial date and time are user-defined in the serial terminal interface using: `setdate <yyyy-mm-dd> <hh:mm:ss>`. Otherwise, the RP2040 requires an internet connection to synchronize to the NTP server.

For more information regarding the serial terminal interface, refer to [SD Card](../SD_Card/SD_Card.html).

## Code

All `SD_Card_Methane` library code is in [this git repository]. The code for the serial terminal user interface is in `sd_card_serial.c` and is organized in protothreads for modularity. 

### Includes

The first lines of code in the C source file include header files. *Don't forget to link these in the CMakeLists.txt file!*

API's associated with datetime and Real Time Clock on the RP2040 are used when logging data to the file.
```
#include "pico/util/datetime.h"
#include "hardware/rtc.h"
```

### `main()`

The file protothread that controls the SD card writing and user interface is assigned to core 1. In particular, it is added to the core 1 function `core1_main` with scheduling priority `SCHED_ROUND_ROBIN`. If there are additional protothreads assigned to core 1, they will each be allocated equal CPU time in the order they are initialized.

### File Protothread

#### Serial Terminal

The serial terminal runs indefinitely when the RP2040 is powered on. At the start of each loop, it gets the current date/time and converts it to a string. The wait time allots sufficient time for `rtc_get_datetime(&t)` to finish executing. Otherwise, a subsequent time-dependent command can exhibit weird behavior.

```
rtc_get_datetime(&t);
datetime_to_str(datetime_str, sizeof(datetime_buf), &t);
sleep_ms(100);
```